This will be a notebook used for prototyping my recommendation service that will be powered by Neo4j and the maximal growth algorithm.

Documentation -- https://github.com/UdayLab/PAMI/blob/66bec6b1cf89588657887194b5dccb25f1d10855/notebooks/frequentPattern/maximal/MaxFPGrowth.ipynb

In [8]:
def install_dependencies(libraries):
  for library in libraries:
    !pip install "{library}"

In [9]:
dependencies = [
    "neo4j",
    "python-dotenv",
    "pami",
]

install_dependencies(dependencies)

In [21]:
import warnings
warnings.filterwarnings('ignore', category=DeprecationWarning)

In [19]:
!pip show pami

Name: pami
Version: 2026.7.1.1
Summary: This software is being developed at the University of Aizu, Aizu-Wakamatsu, Fukushima, Japan
Home-page: https://github.com/udayLab/PAMI
Author: Rage Uday Kiran
Author-email: uday.rage@gmail.com
License: GPLv3
Location: /usr/local/lib/python3.12/dist-packages
Requires: deprecated, discord.py, fastparquet, matplotlib, networkx, numpy, pandas, Pillow, plotly, psutil, resource, sphinx, sphinx-rtd-theme, urllib3, validators
Required-by: 


In [10]:
import dotenv
dotenv.load_dotenv()

True

###Test Transactions

In [ ]:
#test transactions
transactions = [
    ['milk', 'butter', 'bread'],
    ['bread', 'butter'],
    ['bread', 'milk'],
    ['milk', 'butter'],
    ['bread']
]

###MaxFP-Growth Algorithm

In [25]:
from PAMI.frequentPattern.maximal import MaxFPGrowth as alg

# 1. Initialize the algorithm with your transaction file and minimum support count
# (PAMI accepts raw transaction files, meaning no annoying one-hot encoding!)
obj = alg.MaxFPGrowth("transactions.txt", minSup=2, sep=",")

# 2. Run the FP-Max mining pipeline
obj.startMine()

# 3. Get your clean, maximal supersets directly as a dictionary or save them
maximal_itemsets = obj.getPatternsAsDataFrame()

Maximal Frequent patterns were generated successfully using MaxFp-Growth algorithm 


In [32]:
items = obj.getPatterns()

In [37]:
transactions = [key.split("\t")[:-1] for key in items.keys()]

In [38]:
transactions

[['milk', 'butter'], ['bread', 'butter'], ['bread', 'milk']]

## Graphical Representation in Neo4j

We'll use the `neo4j` Python driver to connect to your Neo4j database and load the frequent itemsets. Each unique item will be a node labeled `Item`, and we'll create relationships to represent their co-occurrence within the discovered frequent itemsets. For simplicity, we can create a `Frequents` relationship between items in the same itemset.

Make sure your Neo4j instance is running and you have the connection details (URI, username, password).

In [43]:
from neo4j import GraphDatabase
import os

"""
Class that manages the Neo4j database.
"""
class Neo4j_Manager:
  """
  Constructor for the Neo4j manager object
  """
  def __init__(self):
    # Load Neo4j credentials from environment variables
    self.NEO4J_URI = os.getenv('NEO_CONNECTION_URL')
    self.NEO4J_USERNAME = os.getenv('NEO_DB_ID')
    self.NEO4J_PASSWORD = os.getenv('NEO_PASSWORD')

    # Ensure credentials are set
    if not all([self.NEO4J_URI, self.NEO4J_USERNAME, self.NEO4J_PASSWORD]):
        raise ValueError("Neo4j credentials (NEO4J_URI, NEO4J_USERNAME, NEO4J_PASSWORD) must be set in environment variables or Colab secrets.")

  """
  Returns a driver for the Neo4j database
  """
  def get_driver(self):
    return GraphDatabase.driver(self.NEO4J_URI, auth=(self.NEO4J_USERNAME, self.NEO4J_PASSWORD))

  """
  Clears the database of all data before writing. I would have preferred to drop and create a database, but unforutnately such executive commands
  are not allowed for this sdk.
  """
  def clear_database(self):
    with self.driver.session() as session:
      session.run("MATCH (n) DETACH DELETE n", db="recommendations")

  """
  Helper method to write the frequent itemsets to Neo4j.
  params: tx (transaction)
          itemsets (list[set]) -- list of frequent itemsets
  """
  @staticmethod
  def write_frequent_itemsets_to_neo4j(tx, itemsets):
      # Create nodes for all items and relationships for each itemset
      for itemset in itemsets:
          item_names = list(itemset)
          if len(item_names) == 1:
              # For single items, just create the node
              item_name = item_names[0]
              tx.run("MERGE (i:Item {name: $item_name});", item_name=item_name)
          else:
              # For itemsets with multiple items, create nodes and connect them efficiently
              tx.run(
                  "UNWIND $item_names AS item1_name \n"
                  "UNWIND $item_names AS item2_name \n"
                  "WITH item1_name, item2_name \n"
                  "WHERE item1_name < item2_name // Ensure unique pairs and avoid self-loops \n"
                  "MERGE (i1:Item {name: item1_name}) \n"
                  "MERGE (i2:Item {name: item2_name}) \n"
                  "MERGE (i1)-[:ASSOCIATE]-(i2)",
                  item_names=item_names
              )

  """
  Writes the datasets to Neo4j
  """
  def update_database(self, itemsets):
    self.driver = self.get_driver()
    self.clear_database()

    with self.driver.session() as session:
      session.run("CREATE CONSTRAINT IF NOT EXISTS FOR (i:Item) REQUIRE i.name IS UNIQUE;")
      session.execute_write(self.write_frequent_itemsets_to_neo4j, itemsets)

    self.driver.close()

In [44]:
neo4j = Neo4j_Manager()
neo4j.update_database(transactions)